In [ ]:
from pathlib import Path
import rasterio
import pandas as pd
from wildfire_susceptibility.config.loader import ConfigLoader
from wildfire_susceptibility.viz.charts import plot_nan_coverage, plot_vif_correlation, plot_class_balance
from wildfire_susceptibility.viz.maps import render_all_factor_maps

In [ ]:
cfg_obj = ConfigLoader.load_experiment(
    name = "baseline"
)
cfg = cfg_obj.model_dump(mode="python")
figures_dir = cfg_obj.base.figures_dir

# Temporal EDA

## Seasonal Scope
- Original paper uses only summer months, because most incidents happen during the summer.
- However, this may not be the case for Essex. The paper references sources in which it indicates that an alternative approach is to make sub-models for each season. We must check if this is necessary.

In [ ]:
incidents_df = pd.read_csv("./data/bronze/incidents/OutdoorFIres_2009_2025.csv")
incidents_df["CallDateID"] = pd.to_datetime(incidents_df["CallDateID"], dayfirst = True)

incidents_df["Year"] = incidents_df["CallDateID"].dt.year
incidents_df["Month"] = incidents_df["CallDateID"].dt.month

temp = incidents_df[["Year", "Month"]].copy()

def seasons(x):
    if x in [12, 1, 2]:
        return "WINTER"
    elif x in [3, 4, 5]:
        return "SPRING"
    elif x in [6, 7, 8]:
        return "SUMMER"
    elif x in [9, 10, 11]:
        return "FALL"
    else:
        return "?"


temp["Season"] = temp["Month"].apply(seasons)

In [ ]:
incidents_df["Year"].min(), incidents_df["Year"].max()

In [ ]:
temp["Season"].value_counts() / len(temp)

In [ ]:
season_year_counts = temp.groupby(["Season", "Year"]).size().unstack(fill_value=0)
season_year_counts

In [ ]:
pct_season_year_counts = 100 * season_year_counts / season_year_counts.sum(axis = 0)
pct_season_year_counts

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_long = pct_season_year_counts.reset_index().melt(id_vars="Season", var_name="Year", value_name="Percent (%)")
df_long["Year"] = df_long["Year"].astype(int)

plt.figure(figsize=(10,5))
sns.lineplot(data=df_long, x="Year", y="Percent (%)", hue="Season", marker="o")
plt.title("Seasonal incident counts by year")
plt.xticks(sorted(df_long["Year"].unique()), rotation=45)
plt.tight_layout()
plt.show()

In conclusion, it would be wrong to exclude the spring and fall months, since these three account for 93% of all incidents.

##  Feature Temporal Analysis

### NDVI

In [ ]:
from src.wildfire_susceptibility.features.vegetation import VegetationBuilder

In [ ]:
veg_builder = VegetationBuilder(cfg)
ndvi_arr = veg_builder.eda_debug()

In [ ]:
df = pd.DataFrame(ndvi_arr.reshape((402, -1)))
df = df.dropna(
    how = "any",
    axis = 1
)
df = df.reset_index(
    drop = False,
    names = "day"
)
df = df.melt(
    id_vars = "day",
    var_name = "pixel",
    value_name = "ndvi"
)

In [ ]:
x = df["day"]
y = df["ndvi"]

plt.figure(figsize = (10, 6))
plt.hexbin(x, y, gridsize = 200, cmap = "inferno", mincnt = 1)
plt.colorbar(label = "Count of Series")
plt.title("Hexagonal Bin Density")
plt.show()

## Climate

# Nans analysis

In [ ]:
for season in cfg_obj.seasons.active:
    df = pd.read_csv(cfg_obj.base.model_data_dir / f"dataset_train_{season}.csv")
    feature_cols = [c for c in df.columns if c != "label"]
    feature_arrays = {c: df[c].values for c in feature_cols}
    plot_nan_coverage(feature_arrays, figures_dir, season=season)

In [ ]:
for season in cfg_obj.seasons.active:
    df = pd.read_csv(cfg_obj.base.model_data_dir / f"dataset_train_{season}.csv")    

    feature_cols = [c for c in df.columns if c not in ["label", "_x", "_y", "tas", "tasmin"]]
    plot_vif_correlation(
        df, 
        feature_cols, 
        figures_dir, 
        season=season,
        vif_threshold=10.0, 
        corr_threshold=0.8
    )

In [ ]:
# import numpy as np

# for season in config.seasons.active:
#     with rasterio.open(config.base.output_dir / f"risk_labels_{config.labels.density_method}_{config.labels.classify_method}_{season}_train.tif") as src:
#         before = src.read(1)
#     with rasterio.open(config.base.output_dir / f"risk_labels_clean_{season}_train.tif") as src:
#         after = src.read(1)
#     plot_class_balance(before, after, figures_dir, season=season)

In [ ]:
dem_path = cfg_obj.base.output_dir / "topo_elevation.tif"

static_paths = {
    # Topography
    "elevation": cfg_obj.base.output_dir / f"topo_elevation.tif",
    "slope": cfg_obj.base.output_dir / f"topo_slope.tif",
    "aspect": cfg_obj.base.output_dir / f"topo_aspect.tif",

    # Static Dist
    "d_rivers": cfg_obj.base.output_dir / f"dist_rivers.tif",
    "d_roads": cfg_obj.base.output_dir / f"dist_roads.tif",
    "d_activity": cfg_obj.base.output_dir / f"dist_activity.tif",
    "d_buildings": cfg_obj.base.output_dir / f"dist_activity.tif",
    "land_use": cfg_obj.base.output_dir / f"landuse_class.tif",
}

render_all_factor_maps(static_paths, dem_path, figures_dir)

In [ ]:
seasonal_paths = {}

for season in cfg_obj.seasons.active:
    seasonal_paths.update({
        # Meteo
        "tas": cfg_obj.base.output_dir / f"meteo_tas_{season}_train.tif",
        "tasmax": cfg_obj.base.output_dir / f"meteo_tasmax_{season}_train.tif",
        "tasmin": cfg_obj.base.output_dir / f"meteo_tasmin_{season}_train.tif",
        "hurs": cfg_obj.base.output_dir / f"meteo_hurs_{season}_train.tif",
        "sfcWind": cfg_obj.base.output_dir / f"meteo_sfcWind_{season}_train.tif",
        "rainfall": cfg_obj.base.output_dir / f"meteo_rainfall_{season}_train.tif",

        # Dist
        "d_fires": cfg_obj.base.output_dir / f"dist_fires_{season}.tif",

        # Vegetation
        "ndvi": cfg_obj.base.output_dir / f"ndvi_{season}_train.tif",
    })
    render_all_factor_maps(seasonal_paths, dem_path, figures_dir, season=season)

In [ ]:
dfs = {}
for season in cfg_obj.seasons.active:
    temp = pd.read_csv(cfg_obj.base.model_data_dir / f"dataset_train_{season}.csv")
    feature_cols = [c for c in temp.columns if c != "label"]
    std_df = temp[feature_cols].std()
    mean_df = temp[feature_cols].mean()

    dfs[season] = std_df / mean_df

df = pd.concat(dfs, axis=1)

In [ ]:
df.T